In [1]:
'''Setup & Data Generation (Data)'''

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("✅ Libraries loaded.")

# 1. DATA PHASE
raw_data = [
    {"text": "What is the current price of AAPL?", "intent": "stock_analysis"},
    {"text": "Analyze the moving average for Tesla.", "intent": "stock_analysis"},
    {"text": "Should I buy or sell Microsoft today?", "intent": "stock_analysis"},
    {"text": "How do I reset my account password?", "intent": "user_support"},
    {"text": "My payment failed, please help.", "intent": "user_support"},
    {"text": "Where can I update my billing address?", "intent": "user_support"},
]

df = pd.DataFrame(raw_data)
print(f"📊 Dataset created with {len(df)} records.")
display(df.head())

✅ Libraries loaded.
📊 Dataset created with 6 records.


,text,intent
0,What is the current price of AAPL?,stock_analysis
1,Analyze the moving average for Tesla.,stock_analysis
2,Should I buy or sell Microsoft today?,stock_analysis
3,How do I reset my account password?,user_support
4,"My payment failed, please help.",user_support


In [2]:
'''Text Normalization (Preprocess)'''

# 2. PREPROCESS PHASE
def clean_text(text: str) -> str:
    """Strips special characters and normalizes case."""
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

df['clean_text'] = df['text'].apply(clean_text)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['intent'], test_size=0.3, random_state=42
)

print("🧹 Preprocessing complete. Data split into train/test.")

🧹 Preprocessing complete. Data split into train/test.


In [3]:
'''Model Pipeline (Train)'''

# 3. TRAIN PHASE
print("⚙️ Training Intent Router Model...")

# Initialize vectorizer and classifier
vectorizer = TfidfVectorizer()
classifier = LogisticRegression(random_state=42)

# Fit the vectorizer and transform the training text
X_train_vec = vectorizer.fit_transform(X_train)

# Train the model
classifier.fit(X_train_vec, y_train)

print("✅ Model training complete.")

⚙️ Training Intent Router Model...
✅ Model training complete.


In [4]:
'''Scoring (Evaluate)'''

# 4. EVALUATE PHASE
# Transform test data and predict
X_test_vec = vectorizer.transform(X_test)
predictions = classifier.predict(X_test_vec)

# Calculate metrics
acc = accuracy_score(y_test, predictions)
print(f"🎯 Model Accuracy: {acc * 100:.1f}%\n")

print("📊 Classification Report:")
print(classification_report(y_test, predictions, zero_division=0))

🎯 Model Accuracy: 0.0%

📊 Classification Report:
                precision    recall  f1-score   support

stock_analysis       0.00      0.00      0.00       2.0
  user_support       0.00      0.00      0.00       0.0

      accuracy                           0.00       2.0
     macro avg       0.00      0.00      0.00       2.0
  weighted avg       0.00      0.00      0.00       2.0



In [5]:
'''Packaging for Production (Expose Functions)'''

# 5. EXPOSE PHASE
class IntentRouter:
    """
    Production-ready wrapper for the trained intent classifier.
    Exposes a single `predict_intent` function.
    """
    def __init__(self, trained_vectorizer, trained_classifier):
        self.vectorizer = trained_vectorizer
        self.classifier = trained_classifier
        
    def predict_intent(self, user_query: str) -> dict:
        try:
            # Re-use the preprocessing logic
            cleaned_query = clean_text(user_query)
            vec_query = self.vectorizer.transform([cleaned_query])
            
            # Predict
            intent = self.classifier.predict(vec_query)[0]
            
            # Get confidence score (probability)
            probs = self.classifier.predict_proba(vec_query)[0]
            confidence = max(probs)
            
            return {
                "query": user_query,
                "routed_intent": intent,
                "confidence": float(confidence),
                "status": "success"
            }
        except Exception as e:
            return {"query": user_query, "status": f"error: {str(e)}"}

# Instantiate the router
router = IntentRouter(vectorizer, classifier)

# Test the exposed function
sample_query = "Can you check the RSI indicator for Nvidia?"
result = router.predict_intent(sample_query)

print("🚀 Testing Exposed Function:")
for key, value in result.items():
    print(f" - {key}: {value}")

🚀 Testing Exposed Function:
 - query: Can you check the RSI indicator for Nvidia?
 - routed_intent: user_support
 - confidence: 0.7677305605083196
 - status: success
